In [ ]:
# =============================================================================
# SENTIMENT ANALYSIS PIPELINE
# Using TF-IDF + LinearSVC with GridSearchCV Hyperparameter Tuning
# =============================================================================

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================
# Load your preprocessed dataset.
# Expected columns: 'text' (cleaned comment) and 'sentiment' (Positive/Negative/Neutral)

print("=" * 65)
print("STEP 1: Loading Data")
print("=" * 65)

# ── Replace the path below with your actual file path ──────────────────────
df1 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\Airline_train_cleaned.csv')
df2 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\text.tweeet_cleaned.csv')
df3 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\train_cleaned.csv')
df4 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\train_tweet_cleaned.csv')

#Concatenating the dataframes

df = pd.concat([df1, df2, df3, df4], ignore_index=True)   # <-- UPDATE THIS PATH
# ───────────────────────────────────────────────────────────────────────────

print(f"Dataset shape : {df.shape}")
print(f"Label distribution:\n{df['sentiment'].value_counts()}\n")

# Separate features and labels
X = df["text"]
y = df["sentiment"]

# =============================================================================
# STEP 2: TRAIN / TEST SPLIT
# =============================================================================
# Hold out 20 % of data for final evaluation; stratify to preserve class ratios.

print("=" * 65)
print("STEP 2: Train / Test Split  (80 % train | 20 % test)")
print("=" * 65)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}\n")

# =============================================================================
# STEP 3: BUILD THE SKLEARN PIPELINE
# =============================================================================
# The pipeline chains two steps so that GridSearchCV can tune both together:
#   1. tfidf  → TfidfVectorizer   (feature extraction)
#   2. clf    → LinearSVC         (classification)

print("=" * 65)
print("STEP 3: Building the Pipeline  (TF-IDF → LinearSVC)")
print("=" * 65)

pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                sublinear_tf=True,   # Apply log(1 + tf) scaling
                strip_accents="unicode",
                analyzer="word",
                token_pattern=r"\w{2,}",   # Tokens with at least 2 characters
            ),
        ),
        (
            "clf",
            LinearSVC(
                max_iter=2000,       # Enough iterations for convergence
                random_state=42,
            ),
        ),
    ]
)

print("Pipeline created successfully.\n")

# =============================================================================
# STEP 4: DEFINE HYPERPARAMETER GRID FOR GRIDSEARCHCV
# =============================================================================
# Prefix parameter names with the pipeline step name + "__"
#   tfidf__ngram_range  → unigrams only vs. unigrams + bigrams
#   tfidf__max_df       → ignore terms appearing in > X % of docs (stopword proxy)
#   tfidf__min_df       → ignore very rare terms
#   clf__C              → regularisation strength of LinearSVC

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],   # Unigrams | Unigrams + Bigrams
    "tfidf__max_df"     : [0.85, 0.90, 0.95], # Upper document-frequency threshold
    "tfidf__min_df"     : [1, 2, 3],          # Minimum document frequency
    "clf__C"            : [0.1, 1.0, 5.0],    # Regularisation (smaller = more)
}

# =============================================================================
# STEP 5: GRIDSEARCHCV WITH CROSS-VALIDATION
# =============================================================================
# cv=5  → 5-fold stratified cross-validation on the training set
# n_jobs=-1 → use all available CPU cores for speed
# scoring='f1_macro' → optimise macro-averaged F1 (handles class imbalance fairly)

print("=" * 65)
print("STEP 5: Running GridSearchCV  (5-fold CV)")
print("This may take several minutes …")
print("=" * 65)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2,
    refit=True,   # Refit the best estimator on the full training set
)

start_time = time.time()
grid_search.fit(X_train, y_train)
elapsed = time.time() - start_time

print(f"\nGridSearchCV completed in {elapsed / 60:.1f} minutes.\n")

# =============================================================================
# STEP 6: BEST PARAMETERS
# =============================================================================

print("=" * 65)
print("STEP 6: Best Hyperparameters Found")
print("=" * 65)

best_params = grid_search.best_params_
best_cv_score = grid_search.best_score_

for param, value in best_params.items():
    print(f"  {param:<30} {value}")

print(f"\n  Best CV F1-macro score : {best_cv_score:.4f}\n")

# =============================================================================
# STEP 7: EVALUATE ON THE HELD-OUT TEST SET
# =============================================================================

print("=" * 65)
print("STEP 7: Evaluation on Test Set")
print("=" * 65)

# The best estimator (already refit on full X_train) is used for prediction
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# ── Individual metrics ──────────────────────────────────────────────────────
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall    = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1        = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"\n  Accuracy          : {accuracy:.4f}  ({accuracy * 100:.2f} %)")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")

# ── Per-class report ────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("  Classification Report (per class):\n")
print(classification_report(y_test, y_pred, zero_division=0))

# ── Confusion Matrix ────────────────────────────────────────────────────────
labels = sorted(y.unique())   # Alphabetical order: Negative, Neutral, Positive
cm = confusion_matrix(y_test, y_pred, labels=labels)

print("─" * 65)
print("  Confusion Matrix:")
print(f"  (rows = Actual | cols = Predicted | order: {labels})\n")

# Pretty-print as a labelled DataFrame
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df.to_string())
print()

# =============================================================================
# STEP 8: SAVE THE BEST MODEL  (optional — requires joblib)
# =============================================================================

try:
    import joblib

    model_path = "sentiment_model.joblib"
    joblib.dump(best_model, model_path)
    print("=" * 65)
    print(f"STEP 8: Model saved to '{model_path}'")
    print("        Load later with: model = joblib.load('sentiment_model.joblib')")
    print("=" * 65)
except Exception as exc:
    print(f"(Model not saved — {exc})")

print("\nDone.")